# **XGBoost 성능 평가**

---

1. **데이터 로드 및 전처리**
   - `train`, `val`, `test` CSV 파일 읽기
   - `일시`를 `datetime`으로 변환
   - `build_table()`로 시계열 특징 생성  
     → `add_time_feats`, `add_lags`, `add_deltas`, `make_next24_target`

2. **지역별 스케일링 학습**
   - 입력 특징: `StandardScaler → MinMaxScaler`
   - 타깃(`target_next_24h`): `log1p` 후 `StandardScaler`
   - `region_scalers`에 `(입력표준화, 입력정규화, 타깃표준화)` 저장

3. **데이터 스케일 적용**
   - `apply_transform()`으로 train, val, test 전 구간에 스케일러 적용

4. **시간대별 편향 보정**
   - `compute_hour_bias()`로 `val` 구간의 시간(`hour`)별 평균 잔차 계산
   - `apply_hour_bias()`로 `test` 예측 결과 보정  
     (보정량은 `CALIB_CLIP` 비율로 제한)

5. **발전소 단위 평가**
   - 발전소별 `best.json` 로드  
   - 예측 → 역변환(`inverse_target`) → 보정 → `metrics_mwh` 계산  
   - 결과 테이블 생성 및 가중 평균 계산

6. **지역 단위 평가**
   - 지역별 `best.json` 로드  
   - 동일한 절차로 성능 계산 및 가중 평균 산출

7. **결과 저장**
   - 발전소 성능: `table_test_plants.csv`
   - 지역 성능: `table_test_regions.csv`
   - HTML 리포트(`test_tables.html`) 자동 생성 및 브라우저 오픈

---

## 2. 주요 함수별 역할

| 구분 | 함수명 | 기능 요약 |
|------|---------|------------|
| **시계열 특징 생성** | `add_time_feats` | `hour`, `doy` 기반 sin/cos 추가 |
|  | `add_lags` | 그룹별 `lag_k` 생성 |
|  | `add_deltas` | `lag_1 - lag_(k+1)`로 diff 피처 생성 |
|  | `make_next24_target` | 24시간 합계 타깃 생성 |
|  | `build_table` | 위 과정을 통합 수행 |
| **스케일링** | `apply_transform` | 지역별 스케일러 적용 |
|  | `inverse_target` | 예측 결과를 MWh 단위로 복원 |
| **모델 관련** | `load_booster` | XGBoost 모델 로드 |
|  | `plant_model_path` / `region_model_path` | 체크포인트 경로 반환 |
| **평가 및 보정** | `metrics_mwh` | R², RMSE, MAE 계산 |
|  | `compute_hour_bias` | 시간대별 잔차 평균 계산 |
|  | `apply_hour_bias` | 시간대별 보정 적용 |
| **집계 및 출력** | `_weighted_avg` | 표본 수 기반 가중 평균 |
|  | `show_df` | 주피터 / 콘솔용 테이블 표시 |
|  | `write_html` | HTML 보고서 생성 및 자동 오픈 |

---

## 3. 산출물 요약

| 파일명 | 내용 |
|--------|------|
| **table_test_plants.csv** | 발전소별 Test 성능 (R², RMSE, MAE) + 가중 평균 |
| **table_test_regions.csv** | 지역별 Test 성능 (R², RMSE, MAE) + 가중 평균 |
| **test_tables.html** | 위 두 표를 통합한 HTML 리포트 (자동 오픈) |

---

## 4. 주요 파라미터

| 변수명 | 의미 |
|--------|------|
| `FEATURE_MODE` | 입력 피처 세트 크기 (22 또는 48) |
| `USE_PAST_DELTAS` | 과거 변화량(diff) 피처 사용 여부 |
| `USE_HOURLY_CALIB` | 시간대별 예측 보정 활성화 여부 |
| `CALIB_CLIP` | 시간대 보정량의 상한 비율 |
| `CKPT_ROOT` | 체크포인트 저장 루트 경로 |
| `SAVE_DIR` | 결과 저장 폴더 경로 |

In [4]:
# -*- coding: utf-8 -*-
"""
[별도 스크립트] Test 최종 평가표 (HTML 자동 오픈)
- 기존 체크포인트로 Test 성능 계산
- 표 1: 발전소별 성능(Test) + 가중 평균
- 표 2: 지역별 성능(Test) + 가중 평균
- 주피터 표 표시 + HTML/CSV 저장 + 실행 후 자동 웹브라우저 오픈
"""

import os, warnings, webbrowser
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from jinja2 import Template

warnings.filterwarnings("ignore")

# =========================
# 경로와 설정
# =========================
TRAIN_CSV = r"C:\ESG_Project1\file\merge_data\train.csv"
VAL_CSV   = r"C:\ESG_Project1\file\merge_data\val.csv"
TEST_CSV  = r"C:\ESG_Project1\file\merge_data\test.csv"

CKPT_ROOT = r"C:\ESG_Project1\xgboost\output\1w_feat48_rep\checkpoints"
SAVE_DIR  = os.path.join(os.path.dirname(CKPT_ROOT), "Eval_HTML")
os.makedirs(SAVE_DIR, exist_ok=True)

CONTEXT_2W   = False
FEATURE_MODE = 22
USE_DELTAS   = True

USE_HOURLY_CALIB = True
CALIB_CLIP = 0.25

TIME_COL, GROUP_COL, REGION_COL = "일시", "발전구분", "지역"
TARGET_HOURLY = "합산발전량(MWh)"
TARGET_NEXT24 = "target_next_24h"

WEATHER_COLS = ["기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)", "증기압(hPa)",
                "일조(hr)", "일사(MJ/m2)", "적설(cm)", "전운량(10분위)", "중하층운량(10분위)"]
TIME_FEATS = ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]

# =========================
# 주피터 표시 유틸
# =========================
JUPYTER_MODE = True
try:
    from IPython.display import display, HTML
except Exception:
    JUPYTER_MODE = False
    def display(*args, **kwargs): pass
    def HTML(x): return x

def show_df(df: pd.DataFrame, title: str = None):
    if title:
        if JUPYTER_MODE: display(HTML(f"<h3>{title}</h3>"))
        else: print(f"\n[{title}]")
    if JUPYTER_MODE: display(df)
    else:
        try: print(df.to_string(index=False))
        except: print(df.head())

# =========================
# 피처 구성
# =========================
def _lags_for_48():
    return [1,2,3,6,12,18,24,36,48,72,96,120,144,168,240,288,336]

def _resolve_context_lags(context_2w: bool):
    return _lags_for_48() if context_2w else [1,3,6,24]

if FEATURE_MODE == 22:
    LAGS_FEATURE = [1,3,6,24]
    USE_PAST_DELTAS = True if USE_DELTAS else False
elif FEATURE_MODE == 48:
    LAGS_FEATURE = _lags_for_48()
    USE_PAST_DELTAS = True
else:
    raise ValueError("FEATURE_MODE must be 22 or 48")

LAGS_CONTEXT = _resolve_context_lags(CONTEXT_2W)
LAGS_BUILD   = sorted(set(LAGS_CONTEXT).union(LAGS_FEATURE))

def add_time_feats(df):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    df["hour"] = df[TIME_COL].dt.hour
    df["doy"]  = df[TIME_COL].dt.dayofyear
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["doy_sin"]  = np.sin(2*np.pi*df["doy"]/365)
    df["doy_cos"]  = np.cos(2*np.pi*df["doy"]/365)
    return df

def add_lags(df, lags):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    ext = sorted(set(lags + [k+1 for k in lags]))
    for k in ext:
        df[f"lag_{k}"] = df.groupby(GROUP_COL)[TARGET_HOURLY].shift(k)
    return df

def add_deltas(df, lags, use=True):
    if not use: return df
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    for k in lags:
        a, b = "lag_1", f"lag_{k+1}"
        if a in df.columns and b in df.columns:
            df[f"diff_past_{k}"] = df[a] - df[b]
    return df

def make_next24_target(df):
    df = df.copy()
    g = df.groupby(GROUP_COL)[TARGET_HOURLY]
    df[TARGET_NEXT24] = (g.shift(-1).rolling(24, min_periods=24).sum()
                         .reset_index(level=0, drop=True))
    return df

def build_table(df):
    df = add_time_feats(df)
    df = add_lags(df, LAGS_BUILD)
    df = add_deltas(df, LAGS_BUILD, USE_PAST_DELTAS)
    df = make_next24_target(df)
    df.dropna(subset=[TARGET_NEXT24], inplace=True)
    df.fillna(0.0, inplace=True)
    return df

def dmat(X): return xgb.DMatrix(X)

# =========================
# 데이터 로드
# =========================
train_raw = pd.read_csv(TRAIN_CSV, parse_dates=[TIME_COL])
val_raw   = pd.read_csv(VAL_CSV,   parse_dates=[TIME_COL])
test_raw  = pd.read_csv(TEST_CSV,  parse_dates=[TIME_COL])

train_raw = build_table(train_raw)
val_raw   = build_table(val_raw)
test_raw  = build_table(test_raw)

lag_cols  = [f"lag_{k}" for k in LAGS_FEATURE]
diff_cols = [f"diff_past_{k}" for k in LAGS_FEATURE] if USE_PAST_DELTAS else []
all_candidates = WEATHER_COLS + TIME_FEATS + lag_cols + diff_cols
feature_cols = [c for c in all_candidates if c in train_raw.columns]

# =========================
# 스케일러 구성
# =========================
region_scalers = {}
for region, grp in train_raw.groupby(REGION_COL):
    X = grp[feature_cols].to_numpy(np.float32)
    y = np.log1p(grp[[TARGET_NEXT24]].clip(lower=0.0) + 1e-8)
    std = StandardScaler().fit(X)
    mm  = MinMaxScaler().fit(std.transform(X))
    tsc = StandardScaler().fit(y)
    region_scalers[region] = (std, mm, tsc)

def apply_transform(df):
    df = df.copy()
    for region, grp in df.groupby(REGION_COL):
        if region not in region_scalers: continue
        std, mm, tsc = region_scalers[region]
        X = grp[feature_cols].to_numpy(np.float32)
        y = np.log1p(grp[[TARGET_NEXT24]].clip(lower=0.0) + 1e-8)
        df.loc[grp.index, feature_cols]  = mm.transform(std.transform(X))
        df.loc[grp.index, TARGET_NEXT24] = tsc.transform(y)
    return df

train = apply_transform(train_raw)
val   = apply_transform(val_raw)
test  = apply_transform(test_raw)

# =========================
# 메트릭과 보정
# =========================
def inverse_target(region, y_scaled):
    _, _, tsc = region_scalers[region]
    y_log = tsc.inverse_transform(np.asarray(y_scaled).reshape(-1,1)).ravel()
    y = np.expm1(y_log) - 1e-8
    return np.clip(y, 0, None)

def metrics_mwh(region, y_true_scaled, y_pred_scaled, return_series=False):
    y_true = inverse_target(region, y_true_scaled)
    y_pred = inverse_target(region, y_pred_scaled)
    r2  = max(r2_score(y_true, y_pred), 0.0)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    if return_series: return r2, rmse, mae, y_true, y_pred
    return r2, rmse, mae

def _file_ok(p):
    try: return os.path.exists(p) and os.path.getsize(p) > 0
    except: return False

def load_booster(p):
    if not _file_ok(p): return None
    try:
        b = xgb.Booster(); b.load_model(p); return b
    except: return None

def plant_model_path(plant):
    p = os.path.join(CKPT_ROOT, "plant", plant, "best.json")
    return p if _file_ok(p) else None

def region_model_path(region):
    p = os.path.join(CKPT_ROOT, "region", region, "best.json")
    return p if _file_ok(p) else None

def compute_hour_bias(region, booster, df_val_region):
    if df_val_region.empty or booster is None: return {}
    Xv = df_val_region[feature_cols].to_numpy(np.float32)
    yv = df_val_region[TARGET_NEXT24].to_numpy(np.float32)
    y_pred_scaled = booster.predict(dmat(Xv))
    _, _, _, y_true, y_pred = metrics_mwh(region, yv, y_pred_scaled, return_series=True)
    hours = df_val_region["hour"].to_numpy()
    tmp = pd.DataFrame({"hour": hours, "res": y_true - y_pred})
    return tmp.groupby("hour")["res"].mean().to_dict()

def apply_hour_bias(region, y_pred, hours, bias):
    if not USE_HOURLY_CALIB or not bias: return y_pred
    out = y_pred.copy()
    for i, h in enumerate(hours):
        b = bias.get(int(h), 0.0)
        cap = max(1e-8, abs(out[i]) * CALIB_CLIP)
        out[i] = max(0.0, out[i] + max(-cap, min(cap, b)))
    return out

def _weighted_avg(vals: np.ndarray, w: np.ndarray) -> float:
    wsum = float(w.sum()) if float(w.sum()) > 0 else 1.0
    return float((vals * w).sum() / wsum)

# =========================
# Validation에서 지역별 시간대 보정치 계산
# =========================
region_bias = {}
for region, df_va_r in val.groupby(REGION_COL):
    bias = compute_hour_bias(region, load_booster(region_model_path(region)), df_va_r)
    region_bias[region] = bias

# =========================
# 1) 발전소별 성능(Test)
# =========================
plant_rows_test_adj = []
for plant, df_te in test.groupby(GROUP_COL):
    ppath = plant_model_path(plant)
    if not ppath: continue
    booster = load_booster(ppath)
    if booster is None or len(df_te) == 0: continue
    region = df_te[REGION_COL].iloc[0]
    Xte = df_te[feature_cols].to_numpy(np.float32)
    yte = df_te[TARGET_NEXT24].to_numpy(np.float32)
    y_pred_scaled = booster.predict(dmat(Xte))
    _, _, _, y_true_mwh, y_pred_mwh = metrics_mwh(region, yte, y_pred_scaled, return_series=True)
    hours = df_te["hour"].to_numpy()
    bias = region_bias.get(region, {})
    y_pred_adj = apply_hour_bias(region, y_pred_mwh, hours, bias)
    r2 = max(r2_score(y_true_mwh, y_pred_adj), 0.0)
    rmse = float(np.sqrt(mean_squared_error(y_true_mwh, y_pred_adj)))
    mae  = float(mean_absolute_error(y_true_mwh, y_pred_adj))
    plant_rows_test_adj.append({"plant": plant, "region": region, "test_r2": r2, "test_rmse": rmse, "test_mae": mae})

df_plant_test = pd.DataFrame(plant_rows_test_adj)
if not df_plant_test.empty:
    df_plant_test = df_plant_test.sort_values(["region","test_r2"], ascending=[True, False]).reset_index(drop=True)
    w_plant = np.array([len(test[test[GROUP_COL] == p]) for p in df_plant_test["plant"]], dtype=float)
    wr2_plant   = _weighted_avg(df_plant_test["test_r2"].to_numpy(float),   w_plant)
    wrmse_plant = _weighted_avg(df_plant_test["test_rmse"].to_numpy(float), w_plant)
    wmae_plant  = _weighted_avg(df_plant_test["test_mae"].to_numpy(float),  w_plant)
else:
    wr2_plant = wrmse_plant = wmae_plant = float('nan')

plant_table = pd.DataFrame()
if not df_plant_test.empty:
    plant_table = (
        df_plant_test[["plant","region","test_r2","test_rmse","test_mae"]]
        .rename(columns={"plant":"발전소","region":"지역","test_r2":"R²","test_rmse":"RMSE","test_mae":"MAE"})
        .sort_values(["지역","R²"], ascending=[True, False])
        .reset_index(drop=True)
    )
    plant_table = pd.concat([
        plant_table,
        pd.DataFrame([{"발전소":"가중 평균","지역":"","R²":wr2_plant,"RMSE":wrmse_plant,"MAE":wmae_plant}])
    ], ignore_index=True)

# =========================
# 2) 지역별 성능(Test)
# =========================
region_rows_test_post = []
for region, df_te_r in test.groupby(REGION_COL):
    booster_r = load_booster(region_model_path(region))
    if booster_r is None or len(df_te_r) == 0: continue
    Xte = df_te_r[feature_cols].to_numpy(np.float32)
    yte = df_te_r[TARGET_NEXT24].to_numpy(np.float32)
    y_pred_scaled = booster_r.predict(dmat(Xte))
    _, _, _, y_true, y_pred = metrics_mwh(region, yte, y_pred_scaled, return_series=True)
    hours = df_te_r["hour"].to_numpy()
    bias = region_bias.get(region, {})
    y_pred_adj = apply_hour_bias(region, y_pred, hours, bias)
    r2 = max(r2_score(y_true, y_pred_adj), 0.0)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred_adj)))
    mae  = float(mean_absolute_error(y_true, y_pred_adj))
    region_rows_test_post.append({"region": region, "r2": r2, "rmse": rmse, "mae": mae})

df_region_test = pd.DataFrame(region_rows_test_post).sort_values("r2", ascending=False).reset_index(drop=True)

if not df_region_test.empty:
    w_region = np.array([len(test[test[REGION_COL] == r]) for r in df_region_test["region"]], dtype=float)
    wr2_region   = _weighted_avg(df_region_test["r2"].to_numpy(float),  w_region)
    wrmse_region = _weighted_avg(df_region_test["rmse"].to_numpy(float), w_region)
    wmae_region  = _weighted_avg(df_region_test["mae"].to_numpy(float),  w_region)
else:
    wr2_region = wrmse_region = wmae_region = float('nan')

region_table = pd.DataFrame()
if not df_region_test.empty:
    region_table = (
        df_region_test[["region","r2","rmse","mae"]]
        .rename(columns={"region":"지역","r2":"R²","rmse":"RMSE","mae":"MAE"})
        .sort_values("R²", ascending=False)
        .reset_index(drop=True)
    )
    region_table = pd.concat([
        region_table,
        pd.DataFrame([{"지역":"가중 평균","R²":wr2_region,"RMSE":wrmse_region,"MAE":wmae_region}])
    ], ignore_index=True)

# =========================
# 주피터 표시 + CSV 저장
# =========================
if not plant_table.empty:
    show_df(plant_table, "표 1) 발전소별 성능 평가")
    try: plant_table.to_csv(os.path.join(SAVE_DIR, "table_test_plants.csv"), index=False, encoding="utf-8-sig")
    except Exception: pass
else:
    print("⚠ 발전소별 Test 표를 만들 데이터가 없습니다.")

if not region_table.empty:
    show_df(region_table, "표 2) 지역별 성능 평가")
    try: region_table.to_csv(os.path.join(SAVE_DIR, "table_test_regions.csv"), index=False, encoding="utf-8-sig")
    except Exception: pass
else:
    print("⚠ 지역별 Test 표를 만들 데이터가 없습니다.")

# =========================
# HTML 저장 + 자동 오픈
# =========================
def fmt3(v):
    try: return f"{v:.3f}"
    except: return "-"

def write_html(path):
    from jinja2 import Template
    tpl = Template(r"""
<html lang="ko"><head><meta charset="utf-8">
<title>발전소 및 지역별 성능 평가</title>
<style>
body{font-family:Arial, sans-serif; margin:18px;}
table{border-collapse:collapse; width:100%; margin:16px 0;}
th,td{border:1px solid #bbb; padding:6px 8px; text-align:center;}
th{background:#7FFF00;}
h1{margin-top:8px;}
h2{margin-top:28px;}
.bold-row td{font-weight:bold; background:#f9f9f9;}
</style></head>
<body>
<h1 style="text-align:center;">🌟 발전소 및 지역별 성능 평가 🌟</h1>

{% if plant_rows|length > 0 %}
<h2>1️⃣ 발전소별 성능</h2>
<table>
<tr><th>발전소</th><th>지역</th><th>R²</th><th>RMSE</th><th>MAE</th></tr>
{% for r in plant_rows %}
<tr {% if r.is_weighted %}class="bold-row"{% endif %}>
<td>{{r.plant}}</td><td>{{r.region}}</td><td>{{r.r2}}</td><td>{{r.rmse}}</td><td>{{r.mae}}</td>
</tr>
{% endfor %}
</table>
{% endif %}

{% if region_rows|length > 0 %}
<h2>2️⃣ 지역별 성능</h2>
<table>
<tr><th>지역</th><th>R²</th><th>RMSE</th><th>MAE</th></tr>
{% for r in region_rows %}
<tr {% if r.is_weighted %}class="bold-row"{% endif %}>
<td>{{r.region}}</td><td>{{r.r2}}</td><td>{{r.rmse}}</td><td>{{r.mae}}</td>
</tr>
{% endfor %}
</table>
{% endif %}
</body></html>
""")

    # plant rows
    plant_rows = []
    if not plant_table.empty:
        for _, r in plant_table.iterrows():
            plant_rows.append(dict(
                plant=str(r["발전소"]),
                region=str(r["지역"]),
                r2=f"{r['R²']:.3f}" if pd.notna(r["R²"]) else "-",
                rmse=f"{r['RMSE']:.3f}" if pd.notna(r["RMSE"]) else "-",
                mae=f"{r['MAE']:.3f}" if pd.notna(r["MAE"]) else "-",
                is_weighted=(r["발전소"] == "가중 평균")
            ))

    # region rows
    region_rows = []
    if not region_table.empty:
        for _, r in region_table.iterrows():
            region_rows.append(dict(
                region=str(r["지역"]),
                r2=f"{r['R²']:.3f}" if pd.notna(r["R²"]) else "-",
                rmse=f"{r['RMSE']:.3f}" if pd.notna(r["RMSE"]) else "-",
                mae=f"{r['MAE']:.3f}" if pd.notna(r["MAE"]) else "-",
                is_weighted=(r["지역"] == "가중 평균")
            ))

    html = tpl.render(plant_rows=plant_rows, region_rows=region_rows)
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ HTML 저장: {path}")

    # 자동 오픈
    try:
        import webbrowser, os
        webbrowser.open("file://" + os.path.abspath(path))
    except Exception:
        pass



html_path = os.path.join(SAVE_DIR, "test_tables.html")
write_html(html_path)

print("\n🏁 완료")


,발전소,지역,R²,RMSE,MAE
0,남제주소내,고산,0.018734,0.278925,0.229864
1,하동정수장,광양시,0.390319,0.107858,0.082234
2,하동변전소,광양시,0.142494,0.063103,0.052659
3,하동하수처리장,광양시,0.131851,0.075671,0.062633
4,하동보건소,광양시,0.088817,0.058520,0.050194
5,하동공설운동장,광양시,0.000000,0.432225,0.313763
6,하동본부,광양시,0.000000,9.145603,7.631180
7,부산신항,부산,0.280948,0.122941,0.096325
8,부산수처리장,부산,0.187470,0.154436,0.134073
9,부산복합자재창고,부산,0.000000,0.346920,0.289057


,지역,R²,RMSE,MAE
0,울진,0.269463,9.354023,7.858301
1,영월,0.202018,1.689785,1.407445
2,고산,0.017844,0.279052,0.229965
3,부산,0.000000,3.284978,3.090507
4,광양시,0.000000,10.791380,10.478106
5,성산,0.000000,2.270931,2.047147
6,인천,0.000000,2.805377,2.474214
7,가중 평균,0.025754,5.550212,5.228096


✅ HTML 저장: C:\ESG_Project1\xgboost\output\1w_feat48_rep\Eval_HTML\test_tables.html

🏁 완료


# **XGBoost 지역 모델 기반 이상치 탐지 파이프라인 정리**

## 📌 개요
- **목적**: 지역별 `best.json` 모델을 활용해 테스트 구간 예측값을 생성하고,  
  절대오차 상위 `TOP_P`(기본 1%) 데이터를 이상치로 탐지한다.  
- **결과물**:  
  - 지역별 이상치 비율 및 주요 기상 요인 표  
  - 지역별/전체 그래프  
  - HTML 리포트 (`outliers_report.html`)

---

## 🧩 파이프라인 단계

### 1️⃣ 경로 및 스위치 설정
- 입력 데이터: `train.csv`, `test.csv`
- 체크포인트 폴더: `CKPT_ROOT`
- **설정 변수**
  - `TOP_P`: 이상치 상위 비율 (기본 0.01)
  - `FEATURE_MODE`: 22 또는 48
  - `USE_PAST_DELTAS`: 과거 변화량(diff) 사용 여부
  - `LAGS_BUILD`: 컨텍스트 + 대표 lag 합집합

---

### 2️⃣ 데이터 로드 및 테이블 빌드
- **입력**
  - `train.csv`, `test.csv`
- **수행 절차**
  1. 시간 컬럼(`일시`) 파싱
  2. 특징 생성  
     - `add_time_feats()` : 시간 관련 sin/cos 특징  
     - `add_lags()` : lag_k 생성  
     - `add_deltas()` : diff_past_k 생성  
     - `make_next24_target()` : 다음 24시간 합계 타깃 생성  
  3. 결측치 제거 및 0으로 채움
- **출력**  
  `train_raw`, `test_raw`  

---

### 3️⃣ 지역별 스케일링 학습 및 적용
- **스케일링 구성**
  - 입력 X: `StandardScaler → MinMaxScaler`
  - 타깃 y: `log1p → StandardScaler`
- **함수**
  - `transform_df(df)`: 지역별 스케일 적용
  - `inverse_target(region, y_scaled)`: 타깃 역변환
- **출력**
  - `region_scalers = { region: (std, mm, tsc) }`

---

### 4️⃣ 모델 로딩 및 지역별 예측
- **모델 불러오기**
  - `region_model_path(region)`: 지역별 `best.json` 경로 반환
  - `load_booster(path)`: XGBoost 모델 로드
  - `dmat(X)`: 입력을 DMatrix로 변환
- **출력**
  - 예측값 `y_pred_model`

---

### 5️⃣ 예측 스케일 자동 판별 및 시계 보정
- **절차**
  1. `inverse_target()`으로 실제값 복원
  2. 예측 스케일 두 후보 비교  
     - A: 원 단위 그대로  
     - B: 역변환(log1p, 표준화 복원)
  3. 첫 30일 구간 R² 비교로 더 높은 쪽 선택
  4. 시간대별 선형 보정  
     - `np.polyfit(y_pred, y_true, 1)` → a_h, b_h  
     - 보정식: `y' = a_h * y_pred + b_h`

---

### 6️⃣ 이상치 탐지
- **기준**
  - 절대오차 |y_true - y_pred| 상위 `TOP_P` 분위수
- **결과**
  - 이상치 개수, 비율, 임계값(threshold)
- **출력**
  - `row_mask`: 이상치 여부 Boolean 배열

---

### 7️⃣ Top3 기상 요인 산출
- **방법**
  1. `WEATHER_COLS`에 포함된 기상 변수만 선택
  2. 예측값(`y_pred_corr`)과 상관계수 계산
  3. 절댓값 기준 상위 3개 변수명만 추출
- **출력**
  - `"Top3 기상 요인"` 문자열

---

### 8️⃣ 시계열 그래프 생성
- **단계**
  1. 시간별 합산  
     - `s_true`, `s_pred` 시리즈 생성  
  2. 이상치 시각 리스트 추출  
  3. 그래프 생성  
     - `plot_region_like_example()`  
       - 실제: 파란 면적  
       - 예측: 주황 면적  
       - 이상치: 빨간 점 표시  
- **보조 함수**
  - `_set_korean_font()`: 한글 폰트 설정  
  - `_style_axes()`, `_format_date_axis()`: 그래프 꾸미기

---

### 9️⃣ 전체(합계) 그래프 생성
- 모든 지역의 시계열 합산 후  
  `plot_region_like_example(region_name="합계", ...)` 실행

---

### 🔟 요약 표 생성
- 지역별 통계와 합계 행 추가  
- **컬럼**
  - `지역`
  - `표본수`
  - `이상치 개수(비율)`
  - `Top3 기상 요인`

---

### ⓫ HTML 리포트 생성
- **템플릿 구성**
  - 합계 그래프  
  - 지역별 표  
  - 지역별 그래프  
- **함수**
  - `Jinja2.Template()`  
  - HTML 파일 저장 및 `webbrowser.open()`으로 자동 실행

---

## 🧮 사용 함수 정리

| 분류 | 함수명 | 주요 역할 |
|------|---------|-----------|
| 데이터 전처리 | `add_time_feats()` | 시간 기반 sin/cos 특징 생성 |
|  | `add_lags()` | lag_k 생성 |
|  | `add_deltas()` | diff_past_k 생성 |
|  | `make_next24_target()` | 24시간 합계 타깃 생성 |
|  | `build_table()` | 위 함수 통합 전처리 |
| 스케일링 | `transform_df()` | 지역별 스케일 적용 |
|  | `inverse_target()` | 타깃 역변환 |
| 모델 | `region_model_path()` | 지역 모델 경로 반환 |
|  | `load_booster()` | 모델 로드 |
|  | `dmat()` | DMatrix 생성 |
| 예측 보정 | `_safe_r2()` | R² 계산 안전 처리 |
|  | `np.polyfit()` | 시간대별 선형 보정 계수 산출 |
| 이상치 탐지 | `np.quantile()` | 상위 분위 임계값 계산 |
| 그래프 | `plot_region_like_example()` | 실제·예측·이상치 시각화 |
|  | `_set_korean_font()`, `_style_axes()`, `_format_date_axis()` | 그래프 스타일 설정 |
| 리포트 | `show_df()` | 주피터/콘솔 표시 |
|  | `Jinja2.Template()` | HTML 리포트 생성 |

---

## 🧾 최종 산출물
- **그래프**
  - 지역별 PNG (`{region}_outlier_line.png`)
  - 합계 그래프 PNG (`overall_outlier_line.png`)
- **표**
  - 지역별 + 합계 요약 표 (`df_summary`)
- **리포트**
  - `outliers_report.html` (자동 오픈)

---


In [5]:
# -*- coding: utf-8 -*-
"""
XGBoost 지역 모델 기반 이상치 탐지 (Test 기준)
- ckpt 경로의 지역별 best.json 로드
- 이상치: '행(발전소×시간)' 절대오차 상위 TOP_P 퍼센트
- 표: 지역별 (표본수=행수, 이상치 개수/비율, Top3 기상 요인) + '합계'
- 그래프: 전체(합산) + 지역별 (X: Time, Y: 합산 발전량(MWh), Outliers=빨간점, 행 기준)
- 주피터 표시는 display, HTML 리포트 자동 오픈
"""

import os, warnings, webbrowser
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from jinja2 import Template

warnings.filterwarnings("ignore")

# =========================
# 경로 / 설정
# =========================
TRAIN_CSV = r"C:\ESG_Project1\file\merge_data\train.csv"
TEST_CSV  = r"C:\ESG_Project1\file\merge_data\test.csv"
CKPT_ROOT = r"C:\ESG_Project1\xgboost\output\1w_feat48_rep\checkpoints"

REPORT_DIR = os.path.join(os.path.dirname(CKPT_ROOT), "Outliers_HTML")
PLOTS_DIR  = os.path.join(REPORT_DIR, "plots")
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

TOP_P = 0.01  # 상위 3%
TIME_COL, GROUP_COL, REGION_COL = "일시", "발전구분", "지역"
TARGET_HOURLY = "합산발전량(MWh)"
TARGET_NEXT24 = "target_next_24h"

WEATHER_COLS = ["기온(°C)", "강수량(mm)", "풍속(m/s)", "습도(%)", "증기압(hPa)",
                "일조(hr)", "일사(MJ/m2)", "적설(cm)", "전운량(10분위)", "중하층운량(10분위)"]
TIME_FEATS = ["hour_sin", "hour_cos", "doy_sin", "doy_cos"]

CONTEXT_2W = False
FEATURE_MODE = 48  # {22, 48}
USE_PAST_DELTAS = True

def _lags_for_48(): return [1,2,3,6,12,18,24,36,48,72,96,120,144,168,240,288,336]
if FEATURE_MODE == 22:
    LAGS_FEATURE = [1,3,6,24]
elif FEATURE_MODE == 48:
    LAGS_FEATURE = _lags_for_48(); USE_PAST_DELTAS = True
else:
    raise ValueError("FEATURE_MODE must be 22 or 48")

LAGS_CONTEXT = [1,3,6,24] if not CONTEXT_2W else _lags_for_48()
LAGS_BUILD = sorted(set(LAGS_CONTEXT).union(LAGS_FEATURE))

# =========================
# 주피터 표시 헬퍼
# =========================
JUPYTER_MODE = True
try:
    from IPython.display import display, HTML
except Exception:
    JUPYTER_MODE = False
    def display(*args, **kwargs): pass
    def HTML(x): return x

def show_df(df: pd.DataFrame, title: str = None, head=50):
    if title:
        if JUPYTER_MODE: display(HTML(f"<h3>{title}</h3>"))
        else: print(f"\n[{title}]")
    if JUPYTER_MODE: display(df.head(head))
    else:
        try: print(df.head(head).to_string(index=False))
        except: print(df.head(head))

# =========================
# 폰트(한글)
# =========================
def _set_korean_font():
    try: matplotlib.rc("font", family="Malgun Gothic")
    except Exception:
        try: matplotlib.rc("font", family="AppleGothic")
        except Exception: matplotlib.rc("font", family="DejaVu Sans")
    matplotlib.rcParams["axes.unicode_minus"] = False
_set_korean_font()

# =========================
# 색상
# =========================
COLOR_ACTUAL_LINE = "#1f77b4"  # 실제 라인(범례용)
COLOR_PRED_FILL   = "#FF8C00"  # 예측 면적(불투명 주황, blending 방지)
COLOR_OUT         = "#d62728"  # 이상치 점

def _style_axes(ax):
    ax.grid(True, alpha=0.22, linestyle="--", linewidth=0.6)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)

def _format_date_axis(ax):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=6, maxticks=10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    for label in ax.get_xticklabels():
        label.set_rotation(30)
        label.set_horizontalalignment("right")

# =========================
# 그래프 (예측=불투명 면적, 실제=해칭+라인 → 색조합 없음)
# =========================
def plot_region_like_example(region_name, s_true, s_pred, out_times,
                             n_out, ratio_pct, thr_value,
                             save_path=None, dot_size=18):
    # 이상치 좌표
    y_vals = [s_true.get(t, np.nan) for t in out_times]
    y_vals = np.asarray(y_vals, float)
    mask   = ~np.isnan(y_vals)
    x_pts  = pd.to_datetime(out_times, errors="coerce").to_numpy()[mask]
    y_pts  = y_vals[mask]

    fig = plt.figure(figsize=(12.5, 4.6))
    ax  = plt.gca()

   # 실제값 먼저 면적형으로 채우기 (파란색)
    ax.fill_between(
        s_true.index, s_true.values,
        facecolor="#1f77b4", edgecolor="none", alpha=1.0, zorder=0, label="실제값")

    # 예측값을 그 위에 올림 (주황색)
    ax.fill_between(
        s_pred.index, s_pred.values,
        facecolor="#FF7B00", edgecolor="none", alpha=1.0, zorder=1, label="예측값")

    # 3) 이상치: 빨간 점
    if len(x_pts) > 0:
        ax.scatter(x_pts, y_pts, s=dot_size, color=COLOR_OUT, alpha=0.95,
                   edgecolor="black", linewidths=0.5, label=f"이상치 {n_out}", zorder=3)

    # 제목/라벨
    ax.set_title(f"{region_name} | Actual vs Predicted (상위 {int(TOP_P*100)}% 이상치, MWh 단위)")
    ax.set_xlabel("날짜")
    ax.set_ylabel("합산발전량 (MWh)")

    # 범례: 실선 샘플
    legend_elems = [
        Line2D([0], [0], color=COLOR_ACTUAL_LINE, lw=2.0, label="실제값"),
        Line2D([0], [0], color="#FF7B00",         lw=2.0, label="예측값"),
        Line2D([0], [0], marker='o', color='none', markerfacecolor=COLOR_OUT,
               markeredgecolor='black', markersize=7, label=f"이상치({n_out})")
    ]
    ax.legend(handles=legend_elems, loc="upper left", frameon=True, framealpha=0.95)

    # 상단 요약(우상단, 반투명 박스)
    ax.text(
        0.99, 0.98, f"비율: {ratio_pct:.2f}%  |  임계값: {thr_value:.4f}",
        transform=ax.transAxes, ha="right", va="top",
        fontsize=10, color="#333",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, boxstyle="round,pad=0.25")
    )

    _format_date_axis(ax)
    _style_axes(ax)
    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=160)
        plt.close(fig)
    else:
        plt.show()

# =========================
# 피처/타깃 구성
# =========================
def add_time_feats(df):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    df["hour"] = df[TIME_COL].dt.hour
    df["doy"]  = df[TIME_COL].dt.dayofyear
    df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
    df["doy_sin"]  = np.sin(2*np.pi*df["doy"]/365)
    df["doy_cos"]  = np.cos(2*np.pi*df["doy"]/365)
    return df

def add_lags(df, lags):
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    ext = sorted(set(lags + [k+1 for k in lags]))
    for k in ext:
        df[f"lag_{k}"] = df.groupby(GROUP_COL)[TARGET_HOURLY].shift(k)
    return df

def add_deltas(df, lags, use=True):
    if not use: return df
    df = df.copy()
    df.sort_values([GROUP_COL, TIME_COL], inplace=True)
    for k in lags:
        a, b = "lag_1", f"lag_{k+1}"
        if a in df.columns and b in df.columns:
            df[f"diff_past_{k}"] = df[a] - df[b]
    return df

def make_next24_target(df):
    df = df.copy()
    g = df.groupby(GROUP_COL)[TARGET_HOURLY]
    df[TARGET_NEXT24] = (g.shift(-1).rolling(24, min_periods=24).sum()
                         .reset_index(level=0, drop=True))
    return df

def build_table(df):
    df = add_time_feats(df)
    df = add_lags(df, LAGS_BUILD)
    df = add_deltas(df, LAGS_BUILD, USE_PAST_DELTAS)
    df = make_next24_target(df)
    df.dropna(subset=[TARGET_NEXT24], inplace=True)
    df.fillna(0.0, inplace=True)
    return df

# =========================
# 데이터 로드 & 스케일
# =========================
train_raw = pd.read_csv(TRAIN_CSV, parse_dates=[TIME_COL])
test_raw  = pd.read_csv(TEST_CSV,  parse_dates=[TIME_COL])

train_raw = build_table(train_raw)
test_raw  = build_table(test_raw)

lag_cols  = [f"lag_{k}" for k in LAGS_FEATURE]
diff_cols = [f"diff_past_{k}" for k in LAGS_FEATURE] if USE_PAST_DELTAS else []
all_candidates = WEATHER_COLS + TIME_FEATS + lag_cols + diff_cols
feature_cols = [c for c in all_candidates if c in train_raw.columns]

# 지역별 스케일러
region_scalers: dict[str, tuple[StandardScaler, MinMaxScaler, StandardScaler]] = {}
for region, grp in train_raw.groupby(REGION_COL):
    X = grp[feature_cols].to_numpy(np.float32)
    y = np.log1p(grp[[TARGET_NEXT24]].clip(lower=0.0) + 1e-8)
    std = StandardScaler().fit(X)
    mm  = MinMaxScaler().fit(std.transform(X))
    tsc = StandardScaler().fit(y)
    region_scalers[region] = (std, mm, tsc)

def transform_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for region, grp in df.groupby(REGION_COL):
        if region not in region_scalers: continue
        std, mm, tsc = region_scalers[region]
        X = grp[feature_cols].to_numpy(np.float32)
        y = np.log1p(grp[[TARGET_NEXT24]].clip(lower=0.0) + 1e-8)
        df.loc[grp.index, feature_cols]  = mm.transform(std.transform(X))
        df.loc[grp.index, TARGET_NEXT24] = tsc.transform(y)
    return df

test = transform_df(test_raw)

def inverse_target(region: str, y_scaled):
    _, _, tsc = region_scalers[region]
    y_log = tsc.inverse_transform(np.asarray(y_scaled).reshape(-1,1)).ravel()
    y = np.expm1(y_log) - 1e-8
    return np.clip(y, 0, None)

def dmat(X):
    return xgb.DMatrix(X, feature_names=feature_cols)

def region_model_path(region: str):
    p = os.path.join(CKPT_ROOT, "region", region, "best.json")
    return p if os.path.exists(p) and os.path.getsize(p) > 0 else None

def load_booster(p):
    try:
        b = xgb.Booster(); b.load_model(p); return b
    except Exception:
        return None

# =========================
# 이상치 탐지 (행 기준) + 그래프
# =========================
summary_rows = []
region_plot_paths = {}
overall_png = None

overall_true = {}
overall_pred = {}
overall_outlier_times_rows = []   # 행 기준 전체 이상치 시각(중복 허용)

overall_rows_count = 0
overall_out_count  = 0

for region, df_te_r in test.groupby(REGION_COL):
    rpath = region_model_path(region)
    booster_r = load_booster(rpath) if rpath else None
    if booster_r is None or len(df_te_r) == 0:
        continue

    # --- 예측 (행 기준) : 스케일 자동 판별 + 시간대별 선형 보정 ---
    Xte = df_te_r[feature_cols].to_numpy(np.float32)
    yte = df_te_r[TARGET_NEXT24].to_numpy(np.float32)
    y_pred_model = booster_r.predict(dmat(Xte))

    # 1) 타깃은 항상 역변환(MWh)
    y_true_row = inverse_target(region, yte)

    # 2) 모델 출력 스케일 자동 판별
    #   - 후보A: 출력이 이미 MWh
    #   - 후보B: 출력이 log1p+표준화 → 역변환 필요
    y_pred_a = np.clip(y_pred_model, 0, None)
    y_pred_b = inverse_target(region, y_pred_model)

    # 앞 30일을 캘리브레이션 구간으로 사용해 더 좋은 쪽 선택
    t0 = df_te_r[TIME_COL].min()
    calib_end = t0 + pd.Timedelta(days=30)
    calib_idx = (df_te_r[TIME_COL] < calib_end).to_numpy()

    def _safe_r2(y, p):
        try:
            return float(max(-1.0, r2_score(y, p)))
        except Exception:
            return -1.0

    r2_a = _safe_r2(y_true_row[calib_idx], y_pred_a[calib_idx])
    r2_b = _safe_r2(y_true_row[calib_idx], y_pred_b[calib_idx])
    y_pred_row = y_pred_b if r2_b >= r2_a else y_pred_a

    # 3) 시간대별(0~23시) 선형 보정 y' = a_h * y_pred + b_h  (캘리브레이션: 앞 30일)
    df_tmp = pd.DataFrame({
        "t": df_te_r[TIME_COL].to_numpy(),
        "y_true": y_true_row,
        "y_pred": y_pred_row
    })
    df_tmp["t"] = pd.to_datetime(df_tmp["t"], errors="coerce")
    df_tmp["hour"] = df_tmp["t"].dt.hour

    cal = df_tmp[df_tmp["t"] < calib_end].copy()
    coefs = {}
    for h in range(24):
        sub = cal[cal["hour"] == h]
        if len(sub) >= 24:
            a, b = np.polyfit(sub["y_pred"].values, sub["y_true"].values, 1)
        else:
            a, b = 1.0, 0.0
        coefs[h] = (a, b)

    a_arr = np.array([coefs[h][0] for h in df_tmp["hour"]])
    b_arr = np.array([coefs[h][1] for h in df_tmp["hour"]])
    df_tmp["y_pred_corr"] = df_tmp["y_pred"] * a_arr + b_arr

    # 4) 이상치 판정(행 기준) 및 그래프용 집계
    y_pred_row = df_tmp["y_pred_corr"].to_numpy()
    ae_row = np.abs(df_tmp["y_true"].to_numpy() - y_pred_row)

    thr = float(np.quantile(ae_row, 1.0 - TOP_P)) if len(ae_row) else 0.0
    row_mask = ae_row >= thr
    n_out = int(row_mask.sum())
    n_rows = int(len(ae_row))
    ratio = 100.0 * n_out / max(1, n_rows)

    # --- Top3 영향요인(기상요인만, 숫자 미표기) ---
    try:
        X_df = pd.DataFrame(Xte, columns=feature_cols)
        weather_feats = [c for c in WEATHER_COLS if c in X_df.columns]
        if len(weather_feats) > 0:
            # 보정 후 예측값 기준 상관계수
            corr_series = X_df[weather_feats].apply(
                lambda s: np.corrcoef(s.values, y_pred_row)[0, 1]
            ).astype(float)
            corr_series = corr_series.replace([np.inf, -np.inf], np.nan).fillna(0.0)
            top_names = corr_series.abs().sort_values(ascending=False).index[:3].tolist()
            top3 = ", ".join(top_names) if top_names else "-"
        else:
            top3 = "-"
    except Exception:
        top3 = "-"

    # --- 그래프용: 시간별 합산 + 이상치 시각들 ---
    df_tmp["is_out"] = row_mask
    s_true = df_tmp.groupby("t", as_index=True)["y_true"].sum().sort_index()
    s_pred = df_tmp.groupby("t", as_index=True)["y_pred_corr"].sum().sort_index()

    out_times_rows = df_tmp.loc[df_tmp["is_out"], "t"].tolist()  # 중복 허용

    # ---- 그래프 저장 ----
    region_png = os.path.join(PLOTS_DIR, f"{region}_outlier_line.png")
    plot_region_like_example(
        region_name=region,
        s_true=s_true,
        s_pred=s_pred,
        out_times=[pd.to_datetime(t) for t in out_times_rows],
        n_out=n_out,
        ratio_pct=ratio,
        thr_value=thr,
        save_path=region_png
    )
    region_plot_paths[region] = region_png

    # 요약 행
    summary_rows.append({
        "지역": region,
        "표본수": n_rows,
        "이상치 개수(비율)": f"{n_out} ({ratio:.3f}%)",
        "Top3 기상 요인": top3
    })

    # 전체 그래프 누적
    for t, v in s_true.items():  overall_true[t] = overall_true.get(t, 0.0) + float(v)
    for t, v in s_pred.items():  overall_pred[t] = overall_pred.get(t, 0.0) + float(v)
    overall_outlier_times_rows.extend(out_times_rows)

    overall_rows_count += n_rows
    overall_out_count  += n_out

# =========================
# 전체(합산) 그래프
# =========================
if len(overall_true) > 0:
    s_true_all = pd.Series(overall_true).sort_index()
    s_pred_all = pd.Series(overall_pred).sort_index()

    out_t_all_rows = pd.to_datetime(overall_outlier_times_rows, errors="coerce").tolist()
    ratio_all = 100.0 * overall_out_count / max(1, overall_rows_count)

    overall_png = os.path.join(PLOTS_DIR, "overall_outlier_line.png")
    plot_region_like_example(
        region_name="합계",
        s_true=s_true_all,
        s_pred=s_pred_all,
        out_times=out_t_all_rows,
        n_out=overall_out_count,
        ratio_pct=ratio_all,
        thr_value=0.0,
        save_path=overall_png
    )

# =========================
# 표 생성 (지역별 + 합계)
# =========================
df_summary = pd.DataFrame(summary_rows).sort_values("지역").reset_index(drop=True)
if not df_summary.empty:
    total_n = int(df_summary["표본수"].sum())
    total_k = int(df_summary["이상치 개수(비율)"].str.extract(r"(\d+)")[0].astype(int).sum())
    total_ratio = 100.0 * total_k / max(1, total_n)
    sum_row = pd.DataFrame([{
        "지역": "합계",
        "표본수": total_n,
        "이상치 개수(비율)": f"{total_k} ({total_ratio:.3f}%)",
        "Top3 기상 요인": "-"
    }])
    df_summary = pd.concat([df_summary, sum_row], ignore_index=True)
    show_df(df_summary, "요약 표 (지역별 + 합계)", head=100)

# =========================
# HTML 리포트
# =========================
HTML_PATH = os.path.join(REPORT_DIR, "outliers_report.html")

tpl = Template(r"""
<html lang="ko"><head><meta charset="utf-8">
<title>이상치 탐지 리포트</title>
<style>
body{font-family:Arial, sans-serif; margin:18px;}
h1{margin:0 0 6px 0;}
h2{margin-top:28px;}
table{border-collapse:collapse; width:100%; margin:12px 0;}
th,td{border:1px solid #bbb; padding:6px 8px; text-align:center;}
th{background:#7FFF00;}
tr.sum-row td{font-weight:bold;}
img{max-width:100%; height:auto; border:1px solid #e3e3e3; border-radius:6px;}
.small{color:#666; font-size:12px; margin:6px 0 14px 0;}
</style></head>
<body>
<h1 style="text-align:center;">🌟 발전소 및 지역별 이상치 탐지 결과 🌟</h1>

{% if overall_img %}
<h2>1️⃣ 합계 그래프</h2>
<img src="{{overall_img}}" alt="overall">
{% endif %}

{% if rows|length > 0 %}
<h2>2️⃣ 요약 표 (지역별 + 합계)</h2>
<table>
<tr><th>지역</th><th>표본수</th><th>이상치 개수(비율)</th><th>Top3 기상 요인</th></tr>
{% for r in rows %}
<tr class="{{ 'sum-row' if r.region == '합계' else '' }}">
<td>{{r.region}}</td><td>{{r.n}}</td><td>{{r.k}}</td><td>{{r.top3}}</td>
</tr>
{% endfor %}
</table>
{% endif %}

{% if region_imgs|length > 0 %}
<h2>3️⃣ 지역별 그래프</h2>
{% for item in region_imgs %}
<h3>◾{{item.region}}</h3>
<img src="{{item.path}}" alt="{{item.region}}">
{% endfor %}
{% endif %}
</body></html>
""")

rows = []
if not df_summary.empty:
    for _, rr in df_summary.iterrows():
        rows.append(dict(
            region=str(rr["지역"]),
            n=str(rr["표본수"]),
            k=str(rr["이상치 개수(비율)"]),
            top3=str(rr["Top3 기상 요인"])
        ))

region_imgs = [{"region": r, "path": region_plot_paths[r]} for r in sorted(region_plot_paths.keys())]

html = tpl.render(
    overall_img=overall_png,
    rows=rows,
    region_imgs=region_imgs
)
with open(HTML_PATH, "w", encoding="utf-8") as f:
    f.write(html)

try: webbrowser.open("file://"+HTML_PATH)
except Exception: pass

print(f"✅ HTML 저장: {HTML_PATH}")
print(f"📁 그래프 폴더: {PLOTS_DIR}")


,지역,표본수,이상치 개수(비율),Top3 기상 요인
0,고산,10176,102 (1.002%),"전운량(10분위), 중하층운량(10분위), 일조(hr)"
1,광양시,61056,611 (1.001%),"전운량(10분위), 중하층운량(10분위), 습도(%)"
2,부산,50880,509 (1.000%),"중하층운량(10분위), 전운량(10분위), 습도(%)"
3,성산,20352,204 (1.002%),"전운량(10분위), 중하층운량(10분위), 일조(hr)"
4,영월,10176,102 (1.002%),"중하층운량(10분위), 전운량(10분위), 습도(%)"
5,울진,10176,102 (1.002%),"중하층운량(10분위), 전운량(10분위), 일조(hr)"
6,인천,30528,306 (1.002%),"중하층운량(10분위), 기온(°C), 전운량(10분위)"
7,합계,193344,1936 (1.001%),-


✅ HTML 저장: C:\ESG_Project1\xgboost\output\1w_feat48_rep\Outliers_HTML\outliers_report.html
📁 그래프 폴더: C:\ESG_Project1\xgboost\output\1w_feat48_rep\Outliers_HTML\plots
